# E-commerce Intelligence & Decision System

In [26]:
!pip install -q openpyxl plotly streamlit google-genai

import os
import json
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully!")


Libraries loaded successfully!


In [27]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load and clean the dataset

In [28]:
FILE_PATH = "/content/drive/MyDrive/NEXUS/Online Retail.xlsx"

df_raw = pd.read_excel(FILE_PATH)

print("Raw data shape:", df_raw.shape)


Raw data shape: (541909, 8)


In [29]:
def clean_transactions(df):
    data = df.copy()

    # 1. Remove exact duplicate rows
    data = data.drop_duplicates()

    # 2. Remove rows without CustomerID
    data = data.dropna(subset=["CustomerID"])

    # 3. Parse date
    data["InvoiceDate"] = pd.to_datetime(
        data["InvoiceDate"],
        errors="coerce"
    )

    # 4. Remove rows where date couldn't be parsed
    data = data.dropna(subset=["InvoiceDate"])

    # 5. Remove returns / negative quantities
    data = data[data["Quantity"] > 0]

    # 6. Remove zero or negative prices
    data = data[data["UnitPrice"] > 0]

    # 7. Convert CustomerID to integer
    data["CustomerID"] = data["CustomerID"].astype(int)

    return data.reset_index(drop=True)


df_clean = clean_transactions(df_raw)

df_clean["Revenue"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

print("Raw shape:  ", df_raw.shape)
print("Clean shape:", df_clean.shape)


Raw shape:   (541909, 8)
Clean shape: (392692, 9)


## 2. Revenue and profitability

In [30]:
COST_RATE = 0.60

df_clean["Estimated_COGS"] = (
    df_clean["Revenue"] * COST_RATE
)

df_clean["Gross_Profit"] = (
    df_clean["Revenue"] - df_clean["Estimated_COGS"]
)

df_clean["Gross_Margin"] = (
    df_clean["Gross_Profit"] / df_clean["Revenue"]
)


In [31]:
def revenue_trend(df):
    data = df.copy()

    data["Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby("Month")
        .agg(
            Revenue=("Revenue", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique")
        )
        .reset_index()
    )

    return result


revenue_df = revenue_trend(df_clean)


In [32]:
def gross_margin_summary(df):
    total_revenue = df["Revenue"].sum()
    total_cogs = df["Estimated_COGS"].sum()
    gross_profit = total_revenue - total_cogs

    gross_margin = (
        gross_profit / total_revenue
        if total_revenue != 0
        else 0
    )

    return {
        "revenue": total_revenue,
        "cogs": total_cogs,
        "gross_profit": gross_profit,
        "gross_margin": gross_margin
    }


margin_dict = gross_margin_summary(df_clean)


In [33]:
def product_profitability(df):
    result = (
        df.groupby("Description")
        .agg(
            Revenue=("Revenue", "sum"),
            COGS=("Estimated_COGS", "sum"),
            Gross_Profit=("Gross_Profit", "sum")
        )
        .reset_index()
    )

    result["Gross_Margin"] = (
        result["Gross_Profit"] / result["Revenue"]
    )

    return result.sort_values(
        "Revenue",
        ascending=False
    )


product_df = product_profitability(df_clean)


## 3. RFM customer segmentation

In [34]:
def build_rfm(df):
    snapshot_date = (
        df["InvoiceDate"].max()
        + pd.Timedelta(days=1)
    )

    rfm = (
        df.groupby("CustomerID")
        .agg(
            Recency=(
                "InvoiceDate",
                lambda x: (snapshot_date - x.max()).days
            ),
            Frequency=("InvoiceNo", "nunique"),
            Monetary=("Revenue", "sum")
        )
        .reset_index()
    )

    return rfm


In [35]:
def add_rfm_scores(rfm):
    data = rfm.copy()

    data["R_Score"] = pd.qcut(
        data["Recency"],
        4,
        labels=[4, 3, 2, 1]
    )

    data["F_Score"] = pd.qcut(
        data["Frequency"].rank(method="first"),
        4,
        labels=[1, 2, 3, 4]
    )

    data["M_Score"] = pd.qcut(
        data["Monetary"],
        4,
        labels=[1, 2, 3, 4]
    )

    return data


In [36]:
def assign_segment(row):

    r = int(row["R_Score"])
    f = int(row["F_Score"])
    m = int(row["M_Score"])

    if r >= 3 and f >= 3 and m >= 3:
        return "Champions"

    elif r >= 3 and f >= 2:
        return "Loyal Customers"

    elif r <= 2 and f >= 3:
        return "At Risk"

    elif r <= 2 and f <= 2:
        return "Lost Customers"

    else:
        return "Potential Customers"


rfm_df = add_rfm_scores(build_rfm(df_clean))
rfm_df["Segment"] = rfm_df.apply(
    assign_segment,
    axis=1
)


## 4. Cohort retention

In [37]:
def build_cohort_data(df):

    data = df.copy()

    data["Purchase_Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
    )

    first_purchase = (
        data.groupby("CustomerID")["Purchase_Month"]
        .min()
        .rename("Cohort_Month")
    )

    data = data.merge(
        first_purchase,
        on="CustomerID",
        how="left"
    )

    return data


def add_cohort_period(df):

    data = df.copy()

    data["Cohort_Period"] = (
        (data["Purchase_Month"].dt.year -
         data["Cohort_Month"].dt.year) * 12
        +
        (data["Purchase_Month"].dt.month -
         data["Cohort_Month"].dt.month)
        + 1
    )

    return data


def build_retention_table(df):

    data = df.copy()

    cohort_counts = (
        data.groupby(
            ["Cohort_Month", "Cohort_Period"]
        )["CustomerID"]
        .nunique()
        .reset_index()
    )

    cohort_pivot = cohort_counts.pivot(
        index="Cohort_Month",
        columns="Cohort_Period",
        values="CustomerID"
    )

    cohort_size = cohort_pivot.iloc[:, 0]

    retention = cohort_pivot.divide(
        cohort_size,
        axis=0
    ) * 100

    return retention


cohort_data = build_cohort_data(df_clean)
cohort_data = add_cohort_period(cohort_data)
retention_df = build_retention_table(cohort_data)


## 5. Decision / simulation layer

In [38]:
def prepare_price_quantity_data(df):
    data = df.copy()

    data["Month"] = (
        data["InvoiceDate"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby(["StockCode", "Description", "Month"])
        .agg(
            Avg_Price=("UnitPrice", "mean"),
            Quantity=("Quantity", "sum"),
            Revenue=("Revenue", "sum")
        )
        .reset_index()
    )

    return result


def estimate_price_elasticity(df, min_observations=6):
    results = []

    for product, group in df.groupby("StockCode"):

        data = group[
            (group["Avg_Price"] > 0) &
            (group["Quantity"] > 0)
        ].copy()

        if len(data) < min_observations:
            continue

        x = np.log(data["Avg_Price"])
        y = np.log(data["Quantity"])

        slope, intercept = np.polyfit(x, y, 1)

        predicted = intercept + slope * x

        ss_res = ((y - predicted) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum()

        r_squared = (
            1 - ss_res / ss_tot
            if ss_tot != 0
            else np.nan
        )

        results.append({
            "StockCode": product,
            "Description": data["Description"].iloc[0],
            "Elasticity": slope,
            "R_Squared": r_squared,
            "Observations": len(data),
            "Avg_Price": data["Avg_Price"].mean(),
            "Avg_Monthly_Quantity": data["Quantity"].mean()
        })

    return pd.DataFrame(results)


def classify_elasticity(elasticity):

    if pd.isna(elasticity):
        return "Insufficient Data"

    if elasticity < -1:
        return "Elastic"

    elif elasticity > -1 and elasticity < 0:
        return "Inelastic"

    elif elasticity == -1:
        return "Unit Elastic"

    else:
        return "Positive / Anomalous"


def price_what_if(
    elasticity,
    current_price,
    current_quantity,
    price_change_pct
):
    new_price = current_price * (
        1 + price_change_pct / 100
    )

    predicted_quantity = (
        current_quantity *
        (new_price / current_price) ** elasticity
    )

    current_revenue = (
        current_price * current_quantity
    )

    predicted_revenue = (
        new_price * predicted_quantity
    )

    return {
        "Current_Price": current_price,
        "New_Price": new_price,
        "Current_Quantity": current_quantity,
        "Predicted_Quantity": predicted_quantity,
        "Current_Revenue": current_revenue,
        "Predicted_Revenue": predicted_revenue,
        "Revenue_Change_Pct": (
            (predicted_revenue / current_revenue) - 1
        ) * 100
    }


def price_scenario_table(
    elasticity,
    current_price,
    current_quantity,
    changes=[-20, -10, -5, 0, 5, 10, 20]
):

    results = []

    for change in changes:

        result = price_what_if(
            elasticity,
            current_price,
            current_quantity,
            change
        )

        result["Price_Change_Pct"] = change

        results.append(result)

    return pd.DataFrame(results)


def product_contribution(df):

    result = (
        df.groupby(["StockCode", "Description"])
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Gross_Profit=("Gross_Profit", "sum")
        )
        .reset_index()
    )

    result["Gross_Margin"] = (
        result["Gross_Profit"] /
        result["Revenue"]
    )

    return result


def flag_discontinuation_candidates(df):

    data = df.copy()

    low_volume_threshold = data["Quantity"].quantile(0.25)
    low_margin_threshold = data["Gross_Margin"].quantile(0.25)

    data["Low_Volume"] = (
        data["Quantity"] <= low_volume_threshold
    )

    data["Low_Margin"] = (
        data["Gross_Margin"] <= low_margin_threshold
    )

    data["Discontinue_Flag"] = (
        data["Low_Volume"] &
        data["Low_Margin"]
    )

    return data


def assign_product_action(row):

    if row["Low_Volume"] and row["Low_Margin"]:
        return "Discontinue Candidate"

    elif row["Low_Volume"]:
        return "Review Demand"

    elif row["Low_Margin"]:
        return "Review Pricing/Cost"

    else:
        return "Keep"


price_qty_df = prepare_price_quantity_data(df_clean)
elasticity_df = estimate_price_elasticity(price_qty_df)

elasticity_df["Elasticity_Type"] = (
    elasticity_df["Elasticity"]
    .apply(classify_elasticity)
)

contribution_df = product_contribution(df_clean)

discontinuation_df = flag_discontinuation_candidates(
    contribution_df
)

discontinuation_df["Recommended_Action"] = (
    discontinuation_df
    .apply(assign_product_action, axis=1)
)


/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)
/tmp/ipykernel_2349/266182985.py:39: RankWarning: Polyfit may be poorly conditioned
  slope, intercept = np.polyfit(x, y, 1)


## 6. Shared data contract

This is the hand-off between **Person A's analytics engine** and **Person B's NEXUS dashboard / AI Analyst**.


In [39]:
data_contract = {
    "clean_transactions": df_clean,
    "revenue_trend": revenue_df,
    "product_profitability": product_df,
    "rfm_customers": rfm_df,
    "retention": retention_df,
    "gross_margin": margin_dict,
    "price_elasticity": elasticity_df,
    "product_discontinuation": discontinuation_df
}

print("=== DATA CONTRACT ===")
for name, value in data_contract.items():
    if hasattr(value, "shape"):
        print(f"{name}: {value.shape}")
    else:
        print(f"{name}: ready")


=== DATA CONTRACT ===
clean_transactions: (392692, 12)
revenue_trend: (13, 4)
product_profitability: (3877, 5)
rfm_customers: (4338, 8)
retention: (13, 13)
gross_margin: ready
price_elasticity: (2567, 8)
product_discontinuation: (3897, 11)


## 7. Export the shared analytics outputs

In [15]:
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_clean.to_csv(f"{OUTPUT_DIR}/clean_transactions.csv", index=False)
revenue_df.to_csv(f"{OUTPUT_DIR}/revenue_trend.csv", index=False)
product_df.to_csv(f"{OUTPUT_DIR}/product_profitability.csv", index=False)
rfm_df.to_csv(f"{OUTPUT_DIR}/rfm_customers.csv", index=False)
retention_df.to_csv(f"{OUTPUT_DIR}/retention.csv")
elasticity_df.to_csv(f"{OUTPUT_DIR}/price_elasticity.csv", index=False)
discontinuation_df.to_csv(
    f"{OUTPUT_DIR}/product_discontinuation.csv",
    index=False
)

with open(f"{OUTPUT_DIR}/gross_margin.json", "w") as f:
    json.dump(margin_dict, f, indent=4)

print("Analytics outputs exported successfully.")


Analytics outputs exported successfully.


## 8. NEXUS backend

The following cell creates the shared backend used by the Streamlit application.  
It keeps Person A's analytics intact and adds Person B's query routing + AI narration layer.


In [16]:
%%writefile backend.py
import os
import pandas as pd
import numpy as np
from google import genai


def load_and_clean_data(file_path):
    df = pd.read_excel(file_path)
    df = df.drop_duplicates()
    df = df.dropna(subset=["CustomerID"])
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    df = df.dropna(subset=["InvoiceDate"])
    df = df[df["Quantity"] > 0]
    df = df[df["UnitPrice"] > 0]
    df["CustomerID"] = df["CustomerID"].astype(int)
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]
    df["Estimated_COGS"] = df["Revenue"] * 0.60
    df["Gross_Profit"] = df["Revenue"] - df["Estimated_COGS"]
    df["Gross_Margin"] = df["Gross_Profit"] / df["Revenue"]
    return df.reset_index(drop=True)


def build_revenue_data(df):
    data = df.copy()
    data["Month"] = data["InvoiceDate"].dt.to_period("M").astype(str)
    result = (data.groupby("Month").agg(
        Revenue=("Revenue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique")
    ).reset_index())
    result["Revenue_Growth_Pct"] = result["Revenue"].pct_change() * 100
    return result


def build_rfm(df):
    snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
    rfm = (df.groupby("CustomerID").agg(
        Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Revenue", "sum")
    ).reset_index())
    rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4,3,2,1])
    rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1,2,3,4])
    rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1,2,3,4])
    def segment(row):
        r,f,m = int(row["R_Score"]), int(row["F_Score"]), int(row["M_Score"])
        if r >= 3 and f >= 3 and m >= 3: return "Champions"
        if r >= 3 and f >= 2: return "Loyal Customers"
        if r <= 2 and f >= 3: return "At Risk"
        if r <= 2 and f <= 2: return "Lost Customers"
        return "Potential Customers"
    rfm["Segment"] = rfm.apply(segment, axis=1)
    return rfm


def build_cohort_data(df):
    data = df.copy()
    data["Purchase_Month"] = data["InvoiceDate"].dt.to_period("M")
    first_purchase = data.groupby("CustomerID")["Purchase_Month"].min().rename("Cohort_Month")
    data = data.merge(first_purchase, on="CustomerID", how="left")
    data["Cohort_Period"] = (
        (data["Purchase_Month"].dt.year - data["Cohort_Month"].dt.year) * 12
        + (data["Purchase_Month"].dt.month - data["Cohort_Month"].dt.month) + 1
    )
    return data


def build_retention_table(df):
    counts = (df.groupby(["Cohort_Month", "Cohort_Period"])["CustomerID"]
              .nunique().reset_index())
    pivot = counts.pivot(index="Cohort_Month", columns="Cohort_Period", values="CustomerID")
    cohort_size = pivot.iloc[:, 0]
    return pivot.divide(cohort_size, axis=0) * 100


def product_profitability(df):
    result = (df.groupby(["StockCode", "Description"]).agg(
        Revenue=("Revenue", "sum"), Quantity=("Quantity", "sum"),
        Orders=("InvoiceNo", "nunique"), Gross_Profit=("Gross_Profit", "sum")
    ).reset_index())
    result["Gross_Margin"] = result["Gross_Profit"] / result["Revenue"]
    return result.sort_values("Gross_Profit", ascending=False)


def prepare_price_quantity_data(df):
    data = df.copy()
    data["Month"] = data["InvoiceDate"].dt.to_period("M").astype(str)
    return (data.groupby(["StockCode", "Description", "Month"]).agg(
        Avg_Price=("UnitPrice", "mean"), Quantity=("Quantity", "sum"), Revenue=("Revenue", "sum")
    ).reset_index())


def estimate_price_elasticity(df, min_observations=6):
    results = []
    for product, group in df.groupby("StockCode"):
        data = group[(group["Avg_Price"] > 0) & (group["Quantity"] > 0)].copy()
        if len(data) < min_observations: continue
        x, y = np.log(data["Avg_Price"]), np.log(data["Quantity"])
        slope, intercept = np.polyfit(x, y, 1)
        predicted = intercept + slope * x
        ss_res, ss_tot = ((y-predicted)**2).sum(), ((y-y.mean())**2).sum()
        r2 = 1 - ss_res/ss_tot if ss_tot != 0 else np.nan
        results.append({"StockCode":product,"Description":data["Description"].iloc[0],
                        "Elasticity":slope,"R_Squared":r2,"Observations":len(data),
                        "Avg_Price":data["Avg_Price"].mean(),"Avg_Monthly_Quantity":data["Quantity"].mean()})
    return pd.DataFrame(results)


def classify_elasticity(e):
    if pd.isna(e): return "Insufficient Data"
    if e < -1: return "Elastic"
    if -1 < e < 0: return "Inelastic"
    if e == -1: return "Unit Elastic"
    return "Positive / Anomalous"


def product_contribution(df):
    result = (df.groupby(["StockCode", "Description"]).agg(
        Revenue=("Revenue", "sum"), Quantity=("Quantity", "sum"),
        Orders=("InvoiceNo", "nunique"), Gross_Profit=("Gross_Profit", "sum")
    ).reset_index())
    result["Gross_Margin"] = result["Gross_Profit"] / result["Revenue"]
    return result


def flag_discontinuation_candidates(df):
    data = df.copy()
    low_volume = data["Quantity"].quantile(0.25)
    low_margin = data["Gross_Margin"].quantile(0.25)
    data["Low_Volume"] = data["Quantity"] <= low_volume
    data["Low_Margin"] = data["Gross_Margin"] <= low_margin
    data["Discontinue_Flag"] = data["Low_Volume"] & data["Low_Margin"]
    def action(row):
        if row["Low_Volume"] and row["Low_Margin"]: return "Discontinue Candidate"
        if row["Low_Volume"]: return "Review Demand"
        if row["Low_Margin"]: return "Review Pricing/Cost"
        return "Keep"
    data["Recommended_Action"] = data.apply(action, axis=1)
    return data


def price_what_if(elasticity, current_price, current_quantity, price_change_pct):
    new_price = current_price * (1 + price_change_pct/100)
    predicted_quantity = current_quantity * (new_price/current_price) ** elasticity
    current_revenue = current_price * current_quantity
    predicted_revenue = new_price * predicted_quantity
    return {
        "Current_Price": current_price, "New_Price": new_price,
        "Current_Quantity": current_quantity, "Predicted_Quantity": predicted_quantity,
        "Current_Revenue": current_revenue, "Predicted_Revenue": predicted_revenue,
        "Revenue_Change_Pct": (predicted_revenue/current_revenue - 1) * 100
    }


def build_decision_summary(data_contract):
    revenue = data_contract["revenue_trend"].copy()
    latest = revenue.iloc[-1]
    previous = revenue.iloc[-2] if len(revenue) > 1 else None
    rfm = data_contract["rfm_customers"]
    at_risk = rfm[rfm["Segment"] == "At Risk"]
    products = data_contract["product_discontinuation"]
    return {
        "latest_month": latest["Month"],
        "latest_revenue": float(latest["Revenue"]),
        "previous_month": previous["Month"] if previous is not None else None,
        "previous_revenue": float(previous["Revenue"]) if previous is not None else None,
        "revenue_change_pct": float(latest["Revenue_Growth_Pct"]) if pd.notna(latest.get("Revenue_Growth_Pct")) else None,
        "at_risk_customers": int(len(at_risk)),
        "at_risk_revenue": float(at_risk["Monetary"].sum()) if len(at_risk) else 0.0,
        "discontinue_candidates": int((products["Recommended_Action"] == "Discontinue Candidate").sum())
    }


def route_query(query):
    q = query.lower()
    if any(x in q for x in ["what if", "price", "pricing", "increase", "decrease"]) and any(x in q for x in ["revenue", "product", "sell"]):
        return "what_if"
    if any(x in q for x in ["discontinue", "remove", "drop product", "underperforming product"]):
        return "product_action"
    if any(x in q for x in ["profit", "profitable", "margin", "product performance", "products"]):
        return "product_profitability"
    if any(x in q for x in ["recommend", "should i", "what should", "improve", "action", "next step"]):
        return "recommendation"
    if "revenue" in q and any(x in q for x in ["fall", "drop", "trend", "growth", "change", "decline"]):
        return "revenue_trend"
    if any(x in q for x in ["retention", "cohort", "retained", "repeat purchase"]):
        return "cohort"
    if any(x in q for x in ["customer", "segment", "at risk", "champion", "rfm"]):
        return "rfm"
    return "unknown"


def get_data(intent, data_contract):
    if intent == "revenue_trend": return data_contract["revenue_trend"]
    if intent == "rfm":
        rfm = data_contract["rfm_customers"]
        return rfm[rfm["Segment"] == "At Risk"].sort_values("Monetary", ascending=False).head(20)
    if intent == "cohort": return data_contract["retention"]
    if intent == "product_profitability": return data_contract["product_profitability"].head(20)
    if intent == "product_action": return data_contract["product_discontinuation"].query("Recommended_Action != 'Keep'").sort_values("Gross_Profit").head(30)
    if intent == "recommendation": return build_decision_summary(data_contract)
    if intent == "what_if": return data_contract["price_elasticity"].sort_values("R_Squared", ascending=False).head(20)
    return None


def build_recommendations(data_contract):
    summary = build_decision_summary(data_contract)
    recommendations = []
    if summary["revenue_change_pct"] is not None and summary["revenue_change_pct"] < 0:
        recommendations.append({"priority":"HIGH","area":"Revenue","action":"Investigate the latest-month revenue decline and focus on the products and customer groups contributing most to the change."})
    if summary["at_risk_customers"] > 0:
        recommendations.append({"priority":"HIGH","area":"Customers","action":f"Prioritize win-back campaigns for the {summary['at_risk_customers']:,} customers classified as At Risk, especially high-monetary customers."})
    if summary["discontinue_candidates"] > 0:
        recommendations.append({"priority":"MEDIUM","area":"Products","action":f"Review the {summary['discontinue_candidates']:,} products flagged as discontinuation candidates before removing them from the assortment."})
    if not recommendations:
        recommendations.append({"priority":"MEDIUM","area":"Growth","action":"Use the product, customer, and retention views to identify the strongest opportunities for targeted growth."})
    return recommendations


def narrate(query, intent, data):
    context = {
        "revenue_trend":"Monthly revenue, orders, customers, and revenue growth.",
        "rfm":"Customer Recency, Frequency, Monetary values, RFM scores, and segments.",
        "cohort":"Cohort retention percentages by months since first purchase.",
        "product_profitability":"Product revenue, quantity, orders, gross profit, and gross margin.",
        "product_action":"Products flagged for demand, pricing/cost review, or discontinuation consideration.",
        "recommendation":"A computed business-health summary used to prioritize actions.",
        "what_if":"Products with estimated price elasticity and fit quality."
    }.get(intent, "Computed business data.")
    prompt = f"""You are NEXUS, an AI business analyst.
User question: {query}
Analysis type: {intent}
Data interpretation: {context}
Exact computed data:
{data}

Rules:
- Never invent numbers or unsupported facts.
- Treat Python-computed values as the source of truth.
- Explain the finding, the likely business implication, and a practical next action.
- Clearly state uncertainty when the data does not establish causation.
- Keep the answer concise: 4-7 sentences or short bullets.
"""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        return "The analytics were calculated successfully, but the AI explanation is unavailable because GEMINI_API_KEY is not configured."
    try:
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(model="gemini-3.6-flash", contents=prompt)
        return response.text
    except Exception as e:
        return f"The analytics were calculated successfully, but the AI explanation is temporarily unavailable. Gemini error: {type(e).__name__}"


Writing backend.py


## 9. NEXUS Streamlit application

This creates the dashboard and connects it to the shared backend/data contract.


In [17]:
%%writefile app.py
import os
import streamlit as st
import pandas as pd
import plotly.express as px
from backend import (
    load_and_clean_data, build_revenue_data, build_rfm, build_cohort_data,
    build_retention_table, product_profitability, prepare_price_quantity_data,
    estimate_price_elasticity, classify_elasticity, product_contribution,
    flag_discontinuation_candidates, route_query, get_data, narrate,
    build_recommendations
)

st.set_page_config(page_title="NEXUS — AI Business Intelligence", page_icon="✦", layout="wide", initial_sidebar_state="collapsed")
FILE_PATH = "/content/drive/MyDrive/NEXUS/Online Retail.xlsx"

@st.cache_data
def load_data():
    return load_and_clean_data(FILE_PATH)

df_clean = load_data()
revenue_df = build_revenue_data(df_clean)
rfm_df = build_rfm(df_clean)
cohort_data = build_cohort_data(df_clean)
retention_df = build_retention_table(cohort_data)
product_df = product_profitability(df_clean)
price_qty_df = prepare_price_quantity_data(df_clean)
elasticity_df = estimate_price_elasticity(price_qty_df)
if not elasticity_df.empty:
    elasticity_df["Elasticity_Type"] = elasticity_df["Elasticity"].apply(classify_elasticity)
contribution_df = product_contribution(df_clean)
discontinuation_df = flag_discontinuation_candidates(contribution_df)

data_contract = {
    "clean_transactions": df_clean,
    "revenue_trend": revenue_df,
    "product_profitability": product_df,
    "rfm_customers": rfm_df,
    "retention": retention_df,
    "price_elasticity": elasticity_df,
    "product_discontinuation": discontinuation_df
}

st.markdown("""
<style>
header[data-testid="stHeader"] {
    display: none;
}

[data-testid="stToolbar"] {
    display: none;
}
.stApp { background: radial-gradient(circle at 10% 10%, rgba(99,102,241,.12), transparent 30%), radial-gradient(circle at 90% 20%, rgba(168,85,247,.10), transparent 30%), #08090d; color:#f5f5f7; }
.block-container { max-width:1450px; padding-top:2.5rem; padding-bottom:4rem; }
.brand { font-size:2rem; font-weight:700; letter-spacing:-1px; }
.brand-sub { color:#8b8d98; font-size:.78rem; letter-spacing:2px; text-transform:uppercase; }
.status { color:#7df2a6; font-size:.78rem; letter-spacing:1px; }
.status-dot { display:inline-block; width:7px; height:7px; border-radius:50%; background:#7df2a6; margin-right:7px; }
.kpi { background:rgba(255,255,255,.035); border:1px solid rgba(255,255,255,.075); border-radius:18px; padding:1.4rem 1.5rem; min-height:125px; }
.kpi-title { color:#858793; font-size:.75rem; text-transform:uppercase; letter-spacing:1.5px; }
.kpi-value { font-size:2rem; font-weight:600; margin-top:10px; }
.kpi-change { color:#7df2a6; font-size:.78rem; margin-top:5px; }
.section-label { color:#777985; font-size:.72rem; font-weight:600; letter-spacing:2px; text-transform:uppercase; margin-bottom:1rem; }
.ai-card { background:linear-gradient(135deg,rgba(99,102,241,.14),rgba(168,85,247,.06)); border:1px solid rgba(139,92,246,.25); border-radius:20px; padding:1.5rem; }
</style>
""", unsafe_allow_html=True)

c1,c2=st.columns([4,1])
with c1:
    st.markdown('<div class="brand">NEXUS</div><div class="brand-sub">AI Business Intelligence · Decision Intelligence</div>', unsafe_allow_html=True)
with c2:
    st.markdown('<div class="status"><span class="status-dot"></span>SYSTEM ONLINE</div>', unsafe_allow_html=True)
st.divider()

st.markdown('<div class="section-label">Business Pulse</div>', unsafe_allow_html=True)
total_revenue=df_clean["Revenue"].sum(); total_orders=df_clean["InvoiceNo"].nunique(); total_customers=df_clean["CustomerID"].nunique()
latest_growth=revenue_df["Revenue_Growth_Pct"].iloc[-1] if len(revenue_df)>1 else None
at_risk_count=int((rfm_df["Segment"]=="At Risk").sum())
cols=st.columns(4)
for col,title,value,sub in [
    (cols[0],"Total Revenue",f"${total_revenue:,.0f}","Calculated from dataset"),
    (cols[1],"Orders",f"{total_orders:,}","Unique invoices"),
    (cols[2],"Customers",f"{total_customers:,}","Unique customers"),
    (cols[3],"At-Risk Customers",f"{at_risk_count:,}","RFM segment")]:
    with col: st.markdown(f'<div class="kpi"><div class="kpi-title">{title}</div><div class="kpi-value">{value}</div><div class="kpi-change">{sub}</div></div>', unsafe_allow_html=True)

st.divider()
left,right=st.columns([1.5,1])
with left:
    st.markdown('<div class="section-label">Revenue Trajectory</div>', unsafe_allow_html=True)
    fig=px.line(revenue_df,x="Month",y="Revenue",markers=True)
    fig.update_layout(template="plotly_dark",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",margin=dict(l=10,r=10,t=20,b=10),height=350)
    st.plotly_chart(fig,use_container_width=True)
with right:
    st.markdown('<div class="section-label">Customer Health</div>', unsafe_allow_html=True)
    seg=rfm_df["Segment"].value_counts().reset_index(); seg.columns=["Segment","Customers"]
    fig=px.pie(seg,names="Segment",values="Customers",hole=.55)
    fig.update_layout(template="plotly_dark",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",margin=dict(l=10,r=10,t=20,b=10),height=350)
    st.plotly_chart(fig,use_container_width=True)

st.divider()
st.markdown('<div class="section-label">Recommended Actions</div>', unsafe_allow_html=True)

recs = build_recommendations(data_contract)

if recs:
    rc = st.columns(min(3, len(recs)))

    for i, r in enumerate(recs):
        with rc[i % len(rc)]:
            st.info(
                f"**{r['priority']} · {r['area']}**\n\n"
                f"{r['action']}"
            )
else:
    st.info("No recommendations available.")

st.divider()
st.markdown('<div class="section-label">Decision Lab · Price What-If</div>', unsafe_allow_html=True)
if not elasticity_df.empty:
    options=elasticity_df.dropna(subset=["Elasticity"]).copy()
    options["Label"]=options["StockCode"].astype(str)+" · "+options["Description"].fillna("").str[:55]
    selected=st.selectbox("Product",options["Label"].tolist())
    row=options[options["Label"]==selected].iloc[0]
    change=st.slider("Price change (%)",-20,20,5)
    current_price=float(row["Avg_Price"]); current_qty=float(row["Avg_Monthly_Quantity"]); e=float(row["Elasticity"])
    new_price=current_price*(1+change/100); predicted_qty=current_qty*(new_price/current_price)**e
    current_rev=current_price*current_qty; predicted_rev=new_price*predicted_qty
    a,b,c=st.columns(3)
    a.metric("Estimated elasticity",f"{e:.2f}")
    b.metric("Projected revenue",f"${predicted_rev:,.0f}",f"{(predicted_rev/current_rev-1)*100:+.1f}%")
    c.metric("Projected quantity",f"{predicted_qty:,.0f}",f"{(predicted_qty/current_qty-1)*100:+.1f}%")
    st.caption("Scenario is an estimate based on the fitted price elasticity; it is not a causal forecast.")

st.divider()
st.markdown('<div class="section-label">Ask the Analyst</div>', unsafe_allow_html=True)
query=st.text_input("Business question",placeholder="What should I do to improve revenue? · Which customers are at risk? · What happens if I increase a product price by 5%?")
if st.button("✦ Analyze"):
    if not query.strip(): st.warning("Enter a business question first.")
    else:
        intent=route_query(query)
        if intent=="unknown":
            st.warning("I can analyze revenue, customers/RFM, retention, product profitability, product actions, recommendations, and price scenarios.")
        else:
            data=get_data(intent,data_contract)
            with st.spinner("Analyzing business data..."):
                answer=narrate(query,intent,data)
            st.markdown('<div class="ai-card"><strong>✦ AI Executive Insight</strong></div>',unsafe_allow_html=True)
            st.write(answer)
            with st.expander("Show computed analysis"):
                st.write(f"Intent: `{intent}`")
                if isinstance(data,pd.DataFrame): st.dataframe(data,use_container_width=True)
                else: st.json(data)

st.divider()
st.markdown('<div class="section-label">Explore Analytics</div>', unsafe_allow_html=True)
t1,t2,t3,t4=st.tabs(["Products","Retention","RFM","Product Actions"])
with t1: st.dataframe(product_df.head(25),use_container_width=True)
with t2: st.dataframe(retention_df.style.format("{:.1f}%"),use_container_width=True)
with t3: st.dataframe(rfm_df.groupby("Segment").agg(Customers=("CustomerID","count"),Revenue=("Monetary","sum")).reset_index(),use_container_width=True)
with t4: st.dataframe(discontinuation_df[discontinuation_df["Recommended_Action"]!="Keep"].sort_values("Gross_Profit").head(50),use_container_width=True)


Writing app.py


In [18]:
print("============================================================")
print("DAY 2 INTEGRATION CHECK")
print("============================================================")
print("Clean transactions:", df_clean.shape)
print("Revenue periods:   ", revenue_df.shape)
print("RFM customers:     ", rfm_df.shape)
print("Cohort table:      ", retention_df.shape)
print("Elasticity rows:   ", elasticity_df.shape)
print("Product actions:   ", discontinuation_df.shape)
print("Data contract keys:", list(data_contract.keys()))
print("============================================================")
print("backend.py created: True")
print("app.py created:     True")
print("============================================================")


DAY 2 INTEGRATION CHECK
Clean transactions: (392692, 12)
Revenue periods:    (13, 4)
RFM customers:      (4338, 8)
Cohort table:       (13, 13)
Elasticity rows:    (2567, 8)
Product actions:    (3897, 11)
Data contract keys: ['clean_transactions', 'revenue_trend', 'product_profitability', 'rfm_customers', 'retention', 'gross_margin', 'price_elasticity', 'product_discontinuation']
backend.py created: True
app.py created:     True


In [19]:
# Start NEXUS Streamlit server

import os
import subprocess
import time

# Stop any old Streamlit process
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

# Start Streamlit on port 8501
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit PID:", process.pid)
print("\n--- Streamlit log ---")
print(open("/content/streamlit.log").read())

Streamlit PID: 3308

--- Streamlit log ---


2026-09-11 08:14:01.363 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.193.240.119:8501




## Day 3 — Decision Intelligence

NEXUS now moves from descriptive analytics to decision support: recommendations, broader query routing, product actions, and price what-if analysis.

In [20]:
# Day 3 integration verification

import os

# build_recommendations lives in backend.py, so import it from the generated backend.
from backend import build_recommendations

print("="*60)
print("DAY 3 INTEGRATION CHECK")
print("="*60)
print("Clean transactions:   ", df_clean.shape)
print("Revenue periods:      ", revenue_df.shape)
print("RFM customers:        ", rfm_df.shape)
print("Cohort table:         ", retention_df.shape)
print("Products:             ", product_df.shape)
print("Elasticity rows:      ", elasticity_df.shape)
print("Product actions:      ", discontinuation_df.shape)
print("Recommendation count: ", len(build_recommendations(data_contract)))
print("backend.py created:    ", os.path.exists("/content/backend.py"))
print("app.py created:        ", os.path.exists("/content/app.py"))
print("="*60)


DAY 3 INTEGRATION CHECK
Clean transactions:    (392692, 12)
Revenue periods:       (13, 4)
RFM customers:         (4338, 8)
Cohort table:          (13, 13)
Products:              (3877, 5)
Elasticity rows:       (2567, 8)
Product actions:       (3897, 11)
Recommendation count:  2
backend.py created:     True
app.py created:         True


## Launch NEXUS

Run the launch cell once. Then use the existing ngrok cell to expose port 8501.

In [21]:
import os

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6KuK7fNNm4ATiZAJoJcO2LX28OgqX8yVuQmH1rydheUSA"

print("Gemini API key configured:", bool(os.environ.get("GEMINI_API_KEY")))

Gemini API key configured: True


In [22]:
# Launch NEXUS on port 8501
!pkill -9 -f streamlit || true
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

^C


In [23]:
# Verify Streamlit
import time

time.sleep(5)

print(open("/content/streamlit.log").read())



2026-09-11 08:14:08.190 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.193.240.119:8501




In [24]:
# Optional: expose NEXUS with ngrok

!pip -q install pyngrok

import subprocess
import time

# Stop every existing Streamlit process
subprocess.run(["pkill", "-9", "-f", "streamlit"], capture_output=True)

time.sleep(2)

# Start exactly one Streamlit server
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

time.sleep(5)

print(open("/content/streamlit.log").read())



2026-09-11 08:14:17.705 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.193.240.119:8501




In [25]:
!pip -q install pyngrok

from pyngrok import ngrok

ngrok.set_auth_token("3IscvjI5bnmu74R1S4DuOH9gaYX_6zXeddQS3Vh3XRZN42bEV")

ngrok.kill()

public_url = ngrok.connect(8501)

print("🚀 NEXUS is live:")
print(public_url)

🚀 NEXUS is live:
NgrokTunnel: "https://unwind-untrained-curdle.ngrok-free.dev" -> "http://localhost:8501"
